# Steps
1. Convert token và lemma raw text to csv to get the index
2. Create a list of lemma_POS
3. Extract sentences with lemma_POS in token and lemma csv into individual files for each lemma_POS
4. Parse lại token bằng Stanza
5. Check lại giữa tag cũ và mới xem tỉ lệ sai POS là bao nhiêu

## Import

In [ ]:
# Auto reload 
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import stanza
import re
import sys
from pathlib import Path
import shutil

sys.path.append('../data_preprocessing')
from utils import open_txt, save_to_txt, search_in_txt, replace_in_txt, return_stanza_parsed_tags

In [ ]:
def convert_org_stanza(org_lemma_pos):
    org_lemma = org_lemma_pos.rsplit('_')[0]
    org_pos = org_lemma_pos.split('_')[-1]

    stanza_lemma = org_lemma
    if org_pos == 'N':
        stanza_pos = 'NOUN'
    elif org_pos == 'V':
        stanza_pos = 'VERB'
    elif org_pos == 'A':
        stanza_pos = 'ADJ'
    
    return stanza_lemma, stanza_pos

In [ ]:
pilot_folder = Path(f'./SemEval_swe/')
pilot_folder_SemEval = Path(f'./SemEval_swe_SemEval/')

## Convert token and lemma raw text to csv to get the index

In [ ]:
corp_nos = [1, 2]
data_types = ['token', 'lemma']

In [ ]:
for corp_no in corp_nos:
    for data_type in data_types:
        raw_file = f'./semeval2020_ulscd_swe/corpus{corp_no}/{data_type}/swe{corp_no}.txt'
        raw_csv_path = f'./semeval2020_ulscd_swe/corpus{corp_no}/{data_type}/swe{corp_no}.csv'

        # Read txt
        with open(raw_file, 'r') as f:
            lines = f.readlines()
            lines = [line.rstrip('\n') for line in lines]
            # lines = [line.strip() for line in lines if line.strip()] 
            # Save to DataFrame and then to CSV
            df = pd.DataFrame(lines)
            df.columns = ['sent']
            df.to_csv(raw_csv_path, index=False, header=True)

In [ ]:
raw_file = './semeval2020_ulscd_swe/corpus1/lemma/swe1.txt'
with open(raw_file, 'r') as f:
    raw_lines = f.readlines()

print("Total lines in txt:", len(raw_lines))
print("Non-empty lines:", sum(1 for l in raw_lines if l.strip()))


## Create a list of lemma_POS

In [ ]:
selected_lemmas = [
    'aktiv_A',
    'annandag_N',
    'antyda_V',
    'bearbeta_V',
    'bedömande_N',
    'beredning_N',
    'blockera_V',
    'bolagsstämma_N',
    'bröllop_N',
    'by_N',
    'central_A',
    'färg_N',
    'förhandling_N',
    'gagn_N',
    'granskare_N',
    'kemisk_A',
    'kokärt_N',
    'konduktör_N',
    'krita_N',
    'ledning_N',
    'medium_N',
    'motiv_N',
    'notis_N',
    'studie_N',
    'undertrycka_V',
    'uppfattning_N',
    'uppfostran_N',
    'uppläggning_N',
    'uträtta_V',
    'vaktmästare_N',
    'vegetation_N'
]

## Re-parse with Stanza

In [ ]:
out_folder_reparsed = f'./SemEval_swe/corpus{corp_no}/reparsed/'
os.makedirs(os.path.dirname(out_folder_reparsed), exist_ok=True)

In [ ]:
stanza.download('de')
nlp = stanza.Pipeline(
        'de',
        processors='tokenize,mwt,pos,lemma,depparse',
        use_gpu=True,
        verbose=False,
        tokenize_no_ssplit=True
    )

In [ ]:
for corp_no in corp_nos:
    token_df = pd.read_csv(f'./SemEval_swe/corpus{corp_no}/token/swe{corp_no}.csv')

    reparsed_sents = [] 

    i = 0
    for sent in token_df['sent']:
        doc = nlp(sent)
        for s in doc.sentences:
            lines = []
            lines.append(f'<s id=swe{corp_no}_{i}>')
            for w in s.words:
                lines.append(
                    f"{w.text}\t{w.lemma}\t{w.upos}\t{w.id}\t{w.head}\t{w.deprel}"
                )
            lines.append("</s>")
            reparsed_sents.append("\n".join(lines))
            i += 1

    # Save reparsed sentences to file
    with open(f'{out_folder_reparsed}/ccoha{corp_no}_reparsed.txt', 'w') as f:
        f.write("\n\n".join(reparsed_sents))

In [ ]:
# Lowercase the lemma form in the reparsed files
for corp_no in [1, 2]:
    reparsed_file = f'./SemEval_swe/corpus{corp_no}/reparsed/swe{corp_no}_reparsed.txt'

    with open(reparsed_file, 'r') as f:
        content = f.read()

    lines = content.split('\n')
    normalised_lines = []
    for line in lines:
        if line.startswith('<s id=') or line.startswith('</s>') or line.strip() == '':
            normalised_lines.append(line)
        else:
            parts = line.split('\t')
            if len(parts) >= 2:
                parts[1] = parts[1].lower()
                normalised_lines.append('\t'.join(parts))
            else:
                normalised_lines.append(line)

    with open(reparsed_file, 'w') as f:
        f.write('\n'.join(normalised_lines))

## Check, quantify the mismatches

In [ ]:
# MAIN FUNCTION TO CHECK FOR MISMACHES

# Create a dictionary to store the report
report_mismatches = {}

for corp_no in [1, 2]:
    report_mismatches[corp_no] = {}

    # Org lemma file
    org_parsed_file = f'./SemEval_swe/corpus{corp_no}/lemma/swe{corp_no}.csv'
    org_df = pd.read_csv(org_parsed_file)

    # Reparsed file
    reparsed_file = f'./SemEval_swe/corpus{corp_no}/reparsed/swe{corp_no}_reparsed.txt'
    with open(reparsed_file, 'r') as f:
        reparsed_content = f.read()
        # Separate sentences
        reparsed_sents = reparsed_content.strip().split("\n\n")
        reparsed_sent_count = len(reparsed_sents)

    # Check no of sentences
    org_df['sent'] = org_df['sent'].fillna('') # Fillna because there are some empty lines
    org_sents = org_df['sent'].tolist() 
    org_sent_count = len(org_sents)

    if org_sent_count != reparsed_sent_count:
        report_mismatches[corp_no]['Mismatched numbers of sentences'] = f"org={org_sent_count}, reparsed={reparsed_sent_count}"
        continue
    
    # Statistics for each selected_lemma
    for selected_lemma in selected_lemmas:
        selected_lemma_base = selected_lemma.rsplit('_', 1)[0] # In other datasets than English, there is no _POS in the target

        # Whole file statistics:
        mismatch_sent = 0
        org_miss_lemma_count = 0
        reparsed_miss_lemma_count = 0
        report_mismatches[corp_no][selected_lemma] = []
        
        # Count selected_lemma_base in file
        pattern = rf'\b{re.escape(selected_lemma_base)}\b'

        org_lemma_count = org_df['sent'].str.count(pattern).sum()
        if org_lemma_count == 0: # Avoid division by zero error if the lemma is not found
            org_lemma_count = 1
        
        # Individual sentence check
        for i in range(org_sent_count):
            org_sent = org_sents[i]
            reparsed_sent = reparsed_sents[i]

            # Count selected_lemma_base occurrences in original sentence
            org_count = len(re.findall(pattern, org_sent))

            # Count selected_lemma occurrences in reparsed sentence
            stanza_lemma, stanza_pos = convert_org_stanza(selected_lemma)
            stanza_format = f'\t{stanza_lemma}\t{stanza_pos}\t'
            reparsed_count = reparsed_sent.count(stanza_format)

            # Return stanza tags for the selected_lemma
            stanza_tags = return_stanza_parsed_tags(reparsed_sent, selected_lemma)

            if org_count != reparsed_count:
                mismatch_sent += 1
                (report_mismatches[corp_no][selected_lemma].append(
                    f"Mismatch sentence:{i}, "
                    f"org={org_count}, "
                    f"reparsed={reparsed_count}, "
                    f"org_sent='{org_sent}, "
                    f"stanza_pos={stanza_tags}"))
                
                if org_count >= reparsed_count:
                    reparsed_miss_lemma_count += (org_count - reparsed_count)
                elif org_count < reparsed_count:
                    org_miss_lemma_count += (reparsed_count - org_count)

        # Whole file statistics:
        if mismatch_sent != 0:
            report_mismatches[corp_no][selected_lemma].append(f"Total mismatched sentences: {mismatch_sent} ({mismatch_sent/org_sent_count*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in original file (compared to org): {org_miss_lemma_count} ({org_miss_lemma_count/ (org_lemma_count)*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in reparsed file (compared to org): {reparsed_miss_lemma_count} ({reparsed_miss_lemma_count/org_lemma_count*100:.2f}%)")

In [ ]:
report_mismatches

In [ ]:
with open('./SemEval_swe/mismatch_report.txt', 'w') as f:
    # Write the report_mismatches dictionary to the file beautifully
    for corp_no in report_mismatches:
        f.write(f"Corpus {corp_no}:\n")
        for selected_lemma in report_mismatches[corp_no]:
            f.write(f"Lemma: {selected_lemma}\n")
            for mismatch in report_mismatches[corp_no][selected_lemma]:
                f.write(f"{mismatch}\n")
            f.write("\n")

## Fix the mismatches

In [ ]:
file_paths = [
    './SemEval_swe/corpus1/reparsed/swe1_reparsed.txt',
    './SemEval_swe/corpus2/reparsed/swe2_reparsed.txt'  
]

org_paths = [
    './semeval2020_ulscd_swe/corpus1/lemma/swe1.txt',
    './semeval2020_ulscd_swe/corpus2/lemma/swe2.txt'
]

### Noun vs PROPN

In [ ]:
N_PROPN_patterns = [
    '\tannandag\tPROPN',
    '\tbedömande\tPROPN',
    '\tberedning\tPROPN',
    '\tbolagsstämma\tPROPN',
    '\tbröllop\tPROPN',
    '\tby\tPROPN',
    '\tfärg\tPROPN',
    '\tförhandling\tPROPN',
    '\tgagn\tPROPN',
    '\tgranskare\tPROPN',
    '\tkokärt\tPROPN',
    '\tkonduktör\tPROPN',
    '\tkrita\tPROPN',
    '\tledning\tPROPN',
    '\tmedium\tPROPN',
    '\tmotiv\tPROPN',
    '\tnotis\tPROPN',
    '\tstudie\tPROPN',
    '\tuppfattning\tPROPN',
    '\tuppfostran\tPROPN',
    '\tuppläggning\tPROPN',
    '\tvaktmästare\tPROPN',
    '\tvegetation\tPROPN',
    ]

for file_path in file_paths:
    content = open_txt(file_path)
    for pattern in N_PROPN_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
N_PROPN_patterns_replacement ={
    r'\tannandag\tPROPN': r'\tannandag\tNOUN',
    r'\tberedning\tPROPN': r'\tberedning\tNOUN',
    r'\tbolagsstämma\tPROPN': r'\tbolagsstämma\tNOUN',
    r'\tbröllop\tPROPN': r'\tbröllop\tNOUN',
    r'\tby\tPROPN': r'\tby\tNOUN',
    r'\tfärg\tPROPN': r'\tfärg\tNOUN',
    r'\tgagn\tPROPN': r'\tgagn\tNOUN',
    r'\tgranskare\tPROPN': r'\tgranskare\tNOUN',
    r'\tkonduktör\tPROPN': r'\tkonduktör\tNOUN',
    r'\tkrita\tPROPN': r'\tkrita\tNOUN',
    r'\tledning\tPROPN': r'\tledning\tNOUN',
    r'\tmedium\tPROPN': r'\tmedium\tNOUN',
    r'\tnotis\tPROPN': r'\tnotis\tNOUN',
    r'\tvaktmästare\tPROPN': r'\tvaktmästare\tNOUN',
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in N_PROPN_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

### Spelling

In [ ]:
spelling_patterns = [
    r'\tbyen\t', # r'\tby\t'
    r'\tbyn\t', # r'\tby\t'
    r'\tbyarne\t', # r'\tby\t'
    r'\tbyarna\t', # r'\tby\t'
    r'\tbyar\t', # r'\tby\t'
    r'\tbyens\t', # r'\tby\t'
    r'\tbyns\t', # r'\tby\t'
    r'\tbyarnes\t', # r'\tby\t'
    r'\tbyarnas\t', # r'\tby\t'
    r'\tbyars\t', # r'\tby\t'

    r'\tförhandlingarne\t', # r'\tförhandling\t'

    r'\tgranskar\t', # r'\tgranskare\t'
    r'\tgranskaren\t', # r'\tgranskare\t'
    r'\tgranskarn\t', # r'\tgranskare\t'
    r'\tgranskarne\t', # r'\tgranskare\t'

    r'\tkokärten\t', # r'\tkokärt\t'
    r'\tkokärter\t', # r'\tkokärt\t'
    r'\tkokärterna\t', # r'\tkokärt\t'
    r'\tkokärtor\t', # r'\tkokärt\t'
    r'\tkokärtorna\t', # r'\tkokärt\t' ---

    r'\tkonduktören\t', # r'\tkonduktör\t'

    r'\tmedie\t', # r'\tmedium\t'

    r'\tstudiers\t', # r'\tstudie\t'

    ]

for file_path in file_paths:
    print('CORPUS')
    content = open_txt(file_path)
    for pattern in spelling_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
spelling_patterns_replacement ={
    r'\tbyen\t': r'\tby\t',
    r'\tbyn\t': r'\tby\t',
    r'\tbyarne\t': r'\tby\t',
    r'\tbyarna\t': r'\tby\t',
    r'\tbyar\t': r'\tby\t',
    r'\tbyens\t': r'\tby\t',
    r'\tbyns\t': r'\tby\t',
    r'\tbyarnes\t': r'\tby\t',
    r'\tbyarnas\t': r'\tby\t',
    r'\tbyars\t': r'\tby\t',

    r'\tförhandlingarne\t': r'\tförhandling\t',

    r'\tgranskar\t': r'\tgranskare\t',
    r'\tgranskaren\t': r'\tgranskare\t',
    r'\tgranskarn\t': r'\tgranskare\t',
    r'\tgranskarne\t': r'\tgranskare\t',

    r'\tkokärten\t': r'\tkokärt\t',
    r'\tkokärter\t': r'\tkokärt\t',
    r'\tkokärterna\t': r'\tkokärt\t',
    r'\tkokärtor\t': r'\tkokärt\t',
    r'\tkokärtorna\t': r'\tkokärt\t',

    r'\tkonduktören\t': r'\tkonduktör\t',

    r'\tmedie\t': r'\tmedium\t',

    r'\tstudiers\t': r'\tstudie\t',
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in spelling_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

## Merge 2 corpus

In [ ]:
merge_output =  pilot_folder / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)
corpus_1 = pilot_folder / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'swe(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'swe1_reparsed.txt'
file2_path = corpus_2 / f'swe2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)

## For SemEval: Unified the POS of the target

In [ ]:
SemEval_file_paths = [
    './SemEval_swe_SemEval/corpus1/reparsed/swe1_reparsed.txt',
    './SemEval_swe_SemEval/corpus2/reparsed/swe2_reparsed.txt'  
]

### N vs V vs A

In [ ]:
# SemEval
NVA_semeval_patterns = [
    r'\tantydd\tADJ', # r'\tantyda\tVERB',
    r'\tantydande\tADJ', # r'\tantyda\tVERB',
    
    r'\tbearbetad\tADJ\t', # r'\tbearbeta\tVERB\t',

    r'\tbedömande\tADJ\t', # r'\tbedömande\tNOUN\t',

    r'\tblockerad\tADJ\t', # r'\tblockera\tVERB\t',
    
    r'\tcentral\tPROPN\t', # r'\tcentral\tADJ\t',
    r'\tcentral\tNOUN\t', # r'\tcentral\tADJ\t',
    r'\tcentral\tADV\t', # r'\tcentral\tADJ\t',

    r'\tundertryckt\tADJ\t', # r'\tundertrycka\tVERB\t',
    r'\tundertryckande\tNOUN\t', # r'\tundertrycka\tVERB\t',

    r'\taktiv\tADV\t', # r'\taktiv\tADJ\t',


    ]

for file_path in SemEval_file_paths:
    print('CORPUS')
    content = open_txt(file_path)
    for pattern in NVA_semeval_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
NVA_semeval_patterns_replacement ={
    r'\tantydd\tADJ': r'\tantyda\tVERB',
    r'\tantydande\tADJ': r'\tantyda\tVERB',
    
    r'\tbearbetad\tADJ\t': r'\tbearbeta\tVERB\t',

    r'\tbedömande\tADJ\t': r'\tbedömande\tNOUN\t',

    r'\tblockerad\tADJ\t': r'\tblockera\tVERB\t',
    
    r'\tcentral\tPROPN\t': r'\tcentral\tADJ\t',
    r'\tcentral\tNOUN\t': r'\tcentral\tADJ\t',
    r'\tcentral\tADV\t': r'\tcentral\tADJ\t',

    r'\tundertryckt\tADJ\t': r'\tundertrycka\tVERB\t',
    r'\tundertryckande\tNOUN\t': r'\tundertrycka\tVERB\t',

    r'\taktiv\tADV\t': r'\taktiv\tADJ\t',
    }

for file_path in SemEval_file_paths:
    content = open_txt(file_path)
    for org, repl in NVA_semeval_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

### Unified the POS

In [ ]:
SemEval_lemmas = [
    'aktiv',
    'annandag',
    'antyda',
    'bearbeta',
    'bedömande',
    'beredning',
    'blockera',
    'bolagsstämma',
    'bröllop',
    'by',
    'central',
    'färg',
    'förhandling',
    'gagn',
    'granskare',
    'kemisk',
    'kokärt',
    'konduktör',
    'krita',
    'ledning',
    'medium',
    'motiv',
    'notis',
    'studie',
    'undertrycka',
    'uppfattning',
    'uppfostran',
    'uppläggning',
    'uträtta',
    'vaktmästare',
    'vegetation'
]

In [ ]:
import re

for file_path in SemEval_file_paths:
    content = open_txt(file_path)

    for lemma in SemEval_lemmas:
        # Match: \tlemma\t(ANY_POS)\t and replace the POS by TAR
        content = re.sub(
            rf'\t{re.escape(lemma)}\t[^\t]+\t',   # any POS between tabs
            f'\t{lemma}\tTAR\t',
            content
        )

    save_to_txt(content, file_path)

## Merge 2 corpus SemEval

In [ ]:
merge_output =  pilot_folder_SemEval / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)

corpus_1 = pilot_folder_SemEval / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder_SemEval / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'swe(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'swe1_reparsed.txt'
file2_path = corpus_2 / f'swe2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)